# 05 — A demanding case: breast cancer

*Notebook 5 of 5 · Support Vector Machines*

This is where the chapter comes together. You will run a kernel SVM on real breast-cancer diagnosis
data the honest way, end to end — and meet both its strength (a strong, well-tuned non-linear
classifier) and its honest limits (it must be standardized, it gives a score rather than a probability,
its threshold cannot rescue a confident mistake, and it does not scale to large data). The same dataset
KNN met (the curse, ch 01), logistic regression read calibrated probabilities from (ch 03), and a
decision tree turned into rules (ch 04) — now meet it with the widest margin.

**Prerequisites**
- NB 1–4 (the whole method); ch 01 NB 5, ch 03 NB 6, ch 04 NB 5 (this same dataset, three ways).
- Module 00: NB 07 (confusion matrix, recall), NB 10 (cross-validation), NB 11 (the `Pipeline`,
  fit-on-train-only).

**What you'll be able to do**
- Run an honest SVM workflow: look, scale, tune by CV on train, read a sealed test once.
- Avoid the scale trap a kernel SVM is acutely vulnerable to.
- Place the SVM on the cross-method spine, and read its errors honestly.
- See when a calibrated probability's threshold *cannot* help, and state the kernel SVM's large-n limit.

In [ ]:
import time

import matplotlib.pyplot as plt
import numpy as np
from sklearn.calibration import CalibratedClassifierCV
from sklearn.datasets import make_classification
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import confusion_matrix
from sklearn.model_selection import (
    GridSearchCV,
    StratifiedKFold,
    cross_val_score,
    train_test_split,
)
from sklearn.neighbors import KNeighborsClassifier
from sklearn.pipeline import Pipeline, make_pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.svm import SVC, LinearSVC
from sklearn.tree import DecisionTreeClassifier

from ml_course import colors, datasets, viz

viz.use_course_style()
SEED = 0
np.random.seed(SEED)
CV = StratifiedKFold(n_splits=5, shuffle=True, random_state=SEED)

df = datasets.load_breast_cancer()
X = df.drop(columns="target")
y = (df["target"] == 0).astype(int)  # malignant = 1 (sklearn encodes malignant as 0)
X_train, X_test, y_train, y_test = train_test_split(
    X.to_numpy(), y.to_numpy(), test_size=0.30, stratify=y, random_state=SEED
)
print(f"shape {X.shape}   malignant {int(y.sum())}/{len(y)} ({y.mean():.0%})")
print(f"train {len(y_train)}   test {len(y_test)}   malignant in test {int(y_test.sum())}")

## Where we are, and the stakes

569 patients, 30 measurements per tumour, 212 malignant. This is the dataset KNN felt the curse on,
logistic regression read calibrated probabilities from, and a decision tree turned into a short rule
set. Now we bring the widest margin to it. Because a missed malignancy is the costly error, we watch
**recall on malignant**, not accuracy alone — and, since an SVM measures distances, we will have to
think about the **feature scales** before anything else.

In [ ]:
fig = viz.plot_class_balance(y.map({0: "benign", 1: "malignant"}))
fig.axes[0].set_title("Class balance (malignant = the costly miss)")
plt.show()

scales = df[["mean area", "mean smoothness"]].describe().loc[["mean", "min", "max"]]
print("two features live on wildly different scales:")
print(scales.round(3))

**Read the figure.** About 37 % of the tumours are malignant — not balanced, so a high accuracy
could still hide missed cancers; we will read recall, not accuracy alone. And the two features printed
below differ by orders of magnitude (`mean area` in the hundreds, `mean smoothness` near 0.1) — a red
flag for any method that measures distances.

## The scale trap returns

An SVM with an RBF kernel scores points by `exp(-gamma * ||x - x'||^2)`: a distance. If one feature
ranges in the hundreds and another near 0.1, the large-range feature dominates every distance and the
rest are ignored — exactly the scale trap KNN hit in Chapter 01. The cure is the same: standardize, and
(to avoid leakage) fit the scaler on the training data only, inside a `Pipeline`.

In [ ]:
raw_cv = cross_val_score(SVC(), X_train, y_train, cv=CV).mean()
std_cv = cross_val_score(make_pipeline(StandardScaler(), SVC()), X_train, y_train, cv=CV).mean()
print(f"RBF SVC   raw features CV {raw_cv:.4f}   standardized CV {std_cv:.4f}")

fig, ax = plt.subplots(figsize=(5.5, 4.5))
bars = ax.bar(
    ["raw features", "standardized"], [raw_cv, std_cv],
    color=[colors.COLORS["muted"], colors.COLORS["model"]],
    edgecolor=colors.COLORS["text"], linewidth=0.6,
)
for bar, value in zip(bars, [raw_cv, std_cv], strict=True):
    ax.text(bar.get_x() + bar.get_width() / 2, value, f"{value:.3f}", ha="center", va="bottom",
            color=colors.COLORS["text"])
ax.set_ylim(0.85, 1.0)
ax.set_ylabel("cross-validated accuracy")
ax.set_title("Scaling is mandatory for an SVM")
ax.grid(False)
plt.show()

**Read the figure.** Standardizing lifts cross-validated accuracy from about 0.91 to 0.96 —
roughly five points, for free, by putting the features on a comparable footing. This is the
chapter's headline limit made concrete: for an SVM, scaling is not optional. Every model from here on
lives inside a `Pipeline(StandardScaler, ...)`.

## Tune honestly

With scaling settled, choose `C` and `gamma` by **cross-validation on the training split**, then read
the **sealed test once** (module 00 NB 10). `GridSearchCV` searches inside the `Pipeline`, so the
scaler is refit on every fold — no leakage.

In [ ]:
grid = {
    "svc__C": [0.1, 1, 10, 100],
    "svc__gamma": ["scale", 0.001, 0.01, 0.1],
    "svc__kernel": ["rbf", "linear"],
}
search = GridSearchCV(Pipeline([("scaler", StandardScaler()), ("svc", SVC())]), grid, cv=CV)
search.fit(X_train, y_train)
best_svc = search.best_estimator_.named_steps["svc"]
print(f"best parameters: {search.best_params_}")
print(f"  CV accuracy on train: {search.best_score_:.4f}")
print(f"  sealed test accuracy: {search.score(X_test, y_test):.4f}")
print(f"  the tuned model rests on {best_svc.n_support_.sum()} support vectors (of {len(y_train)})")

In [ ]:
Cs = [0.1, 1, 10, 100]
gammas = ["scale", 0.001, 0.01, 0.1]


def cv_at(C, g):
    pipe = make_pipeline(StandardScaler(), SVC(C=C, gamma=g))
    return cross_val_score(pipe, X_train, y_train, cv=CV).mean()


heat = np.array([[cv_at(C, g) for g in gammas] for C in Cs])
fig, ax = plt.subplots(figsize=(7, 5))
im = ax.imshow(heat, cmap=colors.CMAP_PROBA, aspect="auto", origin="lower")
ax.set_xticks(range(len(gammas)), [str(g) for g in gammas])
ax.set_yticks(range(len(Cs)), [str(c) for c in Cs])
ax.set_xlabel("gamma")
ax.set_ylabel("C")
mid = (heat.max() + heat.min()) / 2
for i in range(len(Cs)):
    for j in range(len(gammas)):
        shade = colors.COLORS["background"] if heat[i, j] > mid else colors.COLORS["text"]
        ax.text(j, i, f"{heat[i, j]:.3f}", ha="center", va="center", color=shade)
fig.colorbar(im, ax=ax, label="cross-validated accuracy")
ax.set_title("C x gamma CV map on breast cancer (RBF, standardized)")
ax.grid(False)
plt.show()

**Read the figure.** The bright cell (`C = 100`, `gamma = 0.001`) is where the grid search
landed — a wide reach with a firm cost. The surface shows the same bias–variance geography you read in
NB 4, now on a real 30-dimensional problem: too small a `gamma` or `C` underfits, the bright band is the
sweet spot, and overly aggressive settings start to slip.

## Line the methods up

The SVM is the chapter's payoff; the fair question is how it compares to the methods the course already
met on this exact dataset. Same pinned split, same sealed test.

In [ ]:
spine = {
    "KNN (k=5)": make_pipeline(StandardScaler(), KNeighborsClassifier(n_neighbors=5)),
    "decision tree": DecisionTreeClassifier(random_state=SEED),
    "logistic reg.": make_pipeline(StandardScaler(), LogisticRegression(max_iter=5000)),
    "SVM (tuned)": search.best_estimator_,
}
scores = {}
for name, est in spine.items():
    est.fit(X_train, y_train)
    scores[name] = est.score(X_test, y_test)
print("cross-method sealed-test accuracy:", {k: round(v, 4) for k, v in scores.items()})

fig, ax = plt.subplots(figsize=(7, 4.5))
names = list(scores)
bars = ax.bar(
    names, [scores[n] for n in names],
    color=[colors.COLORS["muted"]] * 3 + [colors.COLORS["highlight"]],
    edgecolor=colors.COLORS["text"], linewidth=0.6,
)
for bar, name in zip(bars, names, strict=True):
    ax.text(bar.get_x() + bar.get_width() / 2, scores[name], f"{scores[name]:.3f}",
            ha="center", va="bottom", color=colors.COLORS["text"])
ax.set_ylim(0.85, 1.0)
ax.set_ylabel("sealed-test accuracy")
ax.set_title("Cross-method spine on breast cancer (same split)")
ax.grid(False)
plt.show()

**Read the figure.** On this dataset and this split the tuned SVM is the strongest of the four —
ahead of logistic regression, KNN, and a single tree. That is an honest "competitive, and best here",
not "SVMs win in general": a different dataset or split could reorder the bars. What the SVM buys is a
flexible, well-regularized non-linear boundary on clean, scaled data.

In [ ]:
cm = confusion_matrix(y_test, search.predict(X_test))
fig = viz.plot_confusion_matrix(cm, ["benign", "malignant"])
fig.axes[0].set_title("Tuned SVM — confusion matrix (test)")
plt.show()
recall = cm[1, 1] / cm[1].sum()
print(f"malignant recall {recall:.4f}   ({cm[1, 0]} of {cm[1].sum()} cancers missed, "
      f"{cm[0, 1]} false alarms)")

**Read the figure.** The tuned SVM misses 3 of 64 cancers (recall 0.953) and raises 3 false
alarms — fewer of both than the Chapter 04 tree (`[[95, 12], [4, 60]]`). Three missed malignancies is
not nothing, though; the next question is whether we can trade some precision to catch them.

## A probability, and the limit of the threshold

`SVC` gives a score, not a probability (NB 4); calibrate it with `CalibratedClassifierCV` to get one.
In Chapter 03, lowering logistic regression's threshold caught more cancers at the cost of more false
alarms. Does the same lever work here?

In [ ]:
bp = search.best_params_
calibrated = CalibratedClassifierCV(
    Pipeline([("scaler", StandardScaler()),
              ("svc", SVC(C=bp["svc__C"], gamma=bp["svc__gamma"], kernel=bp["svc__kernel"]))]),
    method="sigmoid", ensemble=False,
).fit(X_train, y_train)
proba = calibrated.predict_proba(X_test)[:, 1]

print("malignant = positive; sliding the threshold on the calibrated probability:")
for thr in (0.5, 0.3, 0.2):
    cmt = confusion_matrix(y_test, (proba >= thr).astype(int))
    print(f"  threshold {thr}: recall {cmt[1, 1] / cmt[1].sum():.3f}   "
          f"missed {cmt[1, 0]}   false alarms {cmt[0, 1]}")
missed = (y_test == 1) & (search.predict(X_test) == 0)
print(f"calibrated probability of the 3 missed cancers: {np.sort(proba[missed]).round(3)}")

**Read the result — an honest surprise.** Lowering the threshold does **not** recover the 3
missed cancers — it only piles up false alarms (1 → 7 → 12). Two things are at work. First, lowering a
cut on a fixed score can only *add* positives, never turn a miss into a catch, so the real question is
*how far* below the cut these three sit — and the last line answers it: their calibrated probabilities
are **0.06, 0.13, 0.19**, all under the lowest cut we tried, with a clean gap up to the next true
malignancy. They are not borderline; the SVM is **confidently wrong** about them. Second, the contrast
with Chapter 03: lowering logistic regression's threshold there caught more cancers by recovering a miss
that sat *near* the boundary — the lever reaches borderline cases, not confident ones (a confident miss
stays missed in either model). Catching these three would need a different model or more features, not a
different cutoff.

## The large-n limit

A kernel SVM solves a dense optimization in the number of training samples, so its training time grows
faster than linearly — it does not scale to large data. Rather than quote a complexity class, let us
**measure** it: time the kernel `SVC` against the linear `LinearSVC` as the sample count grows.

In [ ]:
ns = [500, 1000, 2000, 4000, 8000, 16000, 32000]
t_svc, t_lin = [], []
for n in ns:
    Xn, yn = make_classification(n_samples=n, n_features=20, n_informative=10, random_state=SEED)
    Xn = StandardScaler().fit_transform(Xn)
    t0 = time.perf_counter()
    SVC().fit(Xn, yn)
    t_svc.append(time.perf_counter() - t0)
    t0 = time.perf_counter()
    LinearSVC(max_iter=5000).fit(Xn, yn)
    t_lin.append(time.perf_counter() - t0)
exponent = np.polyfit(np.log(ns), np.log(t_svc), 1)[0]
ratio = t_svc[-1] / t_lin[-1]
print(f"kernel-SVC fit time grows about n^{exponent:.2f};")
print(f"  at n=32000: kernel-SVC {t_svc[-1]:.2f}s vs LinearSVC {t_lin[-1]:.3f}s ({ratio:.0f}x)")

fig, ax = plt.subplots(figsize=(7, 5))
ax.plot(ns, t_svc, color=colors.COLORS["highlight"], marker="o", linewidth=2,
        label="kernel SVC (RBF)")
ax.plot(ns, t_lin, color=colors.COLORS["model"], marker="s", linewidth=2, label="LinearSVC")
ax.set_xscale("log")
ax.set_yscale("log")
ax.set_xlabel("n (training samples, log scale)")
ax.set_ylabel("fit time (seconds, log scale)")
ax.set_title("Kernel SVC does not scale; LinearSVC does")
ax.legend()
plt.show()

**Read the figure.** The two lines diverge steadily on the log–log axes: the kernel `SVC` grows
super-linearly (here about `n^1.67`; the worst case is `O(n^3)`), while `LinearSVC` stays nearly flat.
At a few tens of thousands of rows the kernel SVM already takes seconds; at hundreds of thousands or
millions it becomes impractical. For large tabular data, reach for `LinearSVC` / `SGDClassifier` — or a
different family altogether.

## Error analysis, honest limits, and when to use an SVM

A kernel SVM here is a **strong, well-tuned non-linear classifier** — the best of the four on this
problem. But the workflow surfaced its honest limits, every one of them shown rather than asserted:

- it **must be standardized** (raw 0.91 vs scaled 0.96);
- it gives a **score, not a probability** (calibrate for one);
- its **threshold cannot rescue a confident miss** (the 3 cancers sat below probability 0.2);
- it has **no native feature importance** (unlike a tree — a real interpretability cost);
- it **does not scale to large `n`** (measured here: super-linear, ≈ `n^1.6-1.7`; worst case `O(n^3)`).

**When to use it:** clean, well-scaled, small-to-medium data with a clear margin. **When not:** very
large `n`, a need for built-in importances or probabilities, or raw unscaled features you cannot or do
not want to standardize.

## Your turn

1. **Easy — measure the scale gap.** Cross-validate an RBF `SVC` on the raw features and inside a
   `Pipeline(StandardScaler, SVC)`, and report the accuracy gap you find.
2. **Medium — read the map.** From the `C × gamma` heatmap, name the cell you would ship and one you
   would avoid, and say why in terms of bias and variance.
3. **Harder — find the wall.** Grow `n` (replicate or resample rows, or use `make_classification`),
   time `SVC` against `LinearSVC`, and state the `n` at which you would switch to the linear model.

## What you built, and the chapter, end to end

- You ran an honest SVM workflow on real diagnostic data: look → **scale** → tune by CV on train →
  read a sealed test once → analyse errors → state limits.
- You made the **scaling headline** concrete (0.91 → 0.96), placed the SVM on the **cross-method
  spine** (best here, not universally), and read its confusion honestly (3 of 64 missed).
- You met an **honest surprise** — the threshold lever cannot rescue a confident miss — and **measured**
  the kernel SVM's **large-`n` ceiling** (super-linear, ≈ `n^1.6-1.7` this run; worst case `O(n^3)`).

**Vocabulary:** scale sensitivity · CV model selection · cross-method spine · calibrated probability ·
the threshold's limit · the `O(n^2-n^3)` ceiling · `LinearSVC`.

**Support vector machines, end to end.** From the widest street (NB 1) through the soft margin and the
cost `C` (NB 2), the kernel trick (NB 3), and the estimator's knobs `C` / `kernel` / `gamma` (NB 4) to a
real, honestly-evaluated case (NB 5). The SVM is the first **margin-based** method and the home of the
**kernel trick** — powerful on clean, medium-sized data, but scale-sensitive and limited on large `n`.
The course turns next to **ensembles**: Module 06 — Random Forests, which average many decision trees
into a strong, scalable tabular baseline (and fix the single tree's variance you saw in Chapter 04).

## References

- Cortes C, Vapnik V (1995). *Support-vector networks.* Machine Learning 20:273-297.
  DOI: [10.1007/BF00994018](https://doi.org/10.1007/BF00994018)
- Chang C-C, Lin C-J (2011). *LIBSVM.* ACM TIST 2(3):27.
  DOI: [10.1145/1961189.1961199](https://doi.org/10.1145/1961189.1961199) — the solver; the
  `O(n^2-n^3)` training-complexity claim.
- Fan R-E et al. (2008). *LIBLINEAR: a library for large linear classification.* JMLR 9:1871-1874 —
  the solver behind `LinearSVC` (the large-n alternative).
- Street WN, Wolberg WH, Mangasarian OL (1993). *Nuclear feature extraction for breast tumor
  diagnosis.* Proc. SPIE 1905. DOI: [10.1117/12.148698](https://doi.org/10.1117/12.148698) — the WDBC
  dataset.
- Hastie T, Tibshirani R, Friedman J (2009). *The Elements of Statistical Learning*, §12.
  DOI: [10.1007/978-0-387-84858-7](https://doi.org/10.1007/978-0-387-84858-7)

*Previous: 04 — The estimator and its parameters · Next: Module 06 — Random Forests*